# CartPole LIF SNN Reservoir RL Scratchpad

This is a more standard spiking-neural-network version of the CartPole RL experiment. The reservoir uses discrete-time leaky integrate-and-fire neurons with synaptic current decay, membrane decay, threshold/reset spikes, and filtered spike traces.

The task is Gymnasium `CartPole-v1`. The reservoir weights are fixed; a linear actor and linear critic are trained from filtered spike rates, membrane voltage, the current input vector, and a bias. This is a practical readout for an SNN reservoir because filtered spikes alone were too sparse for the linear actor-critic to learn reliably.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, replace
import threading
import time

import gymnasium as gym
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy import sparse
from scipy.sparse.linalg import eigs
from tqdm.auto import tqdm

print(f"gymnasium {gym.__version__}; numpy {np.__version__}")


## Configuration

The reservoir weights are fixed. The only trained parameters are the linear actor and critic readouts.


In [ ]:
@dataclass
class ReservoirConfig:
    n_reservoir: int = 256
    input_dim: int = 10
    rec_fan_in: int = 24
    rec_scale: float = 0.30
    input_scale: float = 1.60
    inhibitory_fraction: float = 0.20
    seed: int = 1

    # Discrete-time LIF dynamics.
    mem_decay: float = 0.90
    syn_decay: float = 0.70
    trace_decay: float = 0.95
    threshold: float = 0.70
    reset_voltage: float = 0.0
    voltage_clip: tuple[float, float] = (-3.0, 3.0)
    voltage_feature_scale: float = 2.0


@dataclass
class RLConfig:
    env_id: str = "CartPole-v1"
    seed: int = 0
    train_episodes: int = 1500
    eval_episodes: int = 50
    eval_every: int = 500
    max_steps: int = 500
    gamma: float = 0.99
    actor_lr: float = 1e-3
    critic_lr: float = 1e-2
    entropy_beta: float = 1e-3
    l2: float = 1e-5
    reward_scale: float = 0.01
    normalize_advantage: bool = True


RESERVOIR_CONFIG = ReservoirConfig()
RL_CONFIG = RLConfig()


## Helpers

The CartPole observation encoder exposes normalized physics variables plus previous action and normalized episode time.


In [ ]:
def moving_average(values, window: int = 50):
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    window = int(max(1, min(window, len(values))))
    kernel = np.ones(window, dtype=np.float32) / window
    prefix = np.full(window - 1, np.nan, dtype=np.float32)
    return np.concatenate([prefix, np.convolve(values, kernel, mode="valid")])


def softmax(logits):
    logits = np.asarray(logits, dtype=np.float32)
    logits = logits - logits.max(axis=-1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=-1, keepdims=True)


def make_env(config, seed: int | None = None):
    env = gym.make(config.env_id)
    if seed is not None:
        env.reset(seed=int(seed))
        env.action_space.seed(int(seed) + 1)
    return env


def encode_cartpole_observation(env, observation, previous_action: float = 0.0, step_fraction: float = 0.0):
    x, x_dot, theta, theta_dot = [float(v) for v in observation]
    theta_limit = float(env.unwrapped.theta_threshold_radians)
    x_limit = float(env.unwrapped.x_threshold)
    return np.array(
        [
            np.clip(x / x_limit, -2.0, 2.0),
            np.clip(x_dot / 3.0, -2.0, 2.0),
            np.clip(theta / theta_limit, -2.0, 2.0),
            np.clip(theta_dot / 3.5, -2.0, 2.0),
            np.sin(theta),
            np.cos(theta),
            float(previous_action),
            float(step_fraction),
            np.clip(abs(theta) / theta_limit, 0.0, 2.0),
            1.0,
        ],
        dtype=np.float32,
    )


## LIF Spiking Reservoir

Each step updates synaptic current, membrane voltage, binary spikes, and an exponentially filtered spike trace. The readout uses filtered spike rates plus membrane voltage. If you want a stricter spike-only readout, remove voltage from `features`, but expect learning to get much harder.


In [ ]:
class Reservoir:
    frame_title = "LIF filtered spike rate"

    def __init__(self, config: ReservoirConfig):
        self.config = config
        self.rng = np.random.default_rng(config.seed)
        self.W_rec = self._make_recurrent_matrix()
        self.W_in = self.rng.normal(
            0.0,
            config.input_scale / np.sqrt(config.input_dim),
            size=(config.n_reservoir, config.input_dim),
        ).astype(np.float32)
        self.reset()

    @property
    def n_features(self):
        # Standard SNN readout features: filtered spikes plus membrane voltage, then input and bias.
        return 2 * self.config.n_reservoir + self.config.input_dim + 1

    def _make_recurrent_matrix(self):
        cfg = self.config
        rows, cols, data = [], [], []
        signs = np.ones(cfg.n_reservoir, dtype=np.float32)
        n_inhibitory = int(round(cfg.inhibitory_fraction * cfg.n_reservoir))
        signs[:n_inhibitory] = -1.0
        self.rng.shuffle(signs)
        self.neuron_signs = signs
        for row in range(cfg.n_reservoir):
            presynaptic = self.rng.choice(cfg.n_reservoir, size=cfg.rec_fan_in, replace=False)
            weights = np.abs(
                self.rng.normal(
                    cfg.rec_scale / np.sqrt(cfg.rec_fan_in),
                    cfg.rec_scale / (2.0 * np.sqrt(cfg.rec_fan_in)),
                    size=cfg.rec_fan_in,
                )
            )
            rows.extend([row] * cfg.rec_fan_in)
            cols.extend(presynaptic.tolist())
            data.extend((signs[presynaptic] * weights).tolist())
        return sparse.csr_matrix(
            (np.asarray(data, dtype=np.float32), (rows, cols)),
            shape=(cfg.n_reservoir, cfg.n_reservoir),
        )

    def reset(self):
        n = self.config.n_reservoir
        self.voltage = np.zeros(n, dtype=np.float32)
        self.current = np.zeros(n, dtype=np.float32)
        self.spikes = np.zeros(n, dtype=np.float32)
        self.trace = np.zeros(n, dtype=np.float32)

    def step(self, input_vector):
        cfg = self.config
        u = np.asarray(input_vector, dtype=np.float32)
        self.current = (cfg.syn_decay * self.current + self.W_rec @ self.spikes + self.W_in @ u).astype(np.float32)
        self.voltage = (cfg.mem_decay * self.voltage + self.current).astype(np.float32)
        self.spikes = (self.voltage >= cfg.threshold).astype(np.float32)
        self.voltage[self.spikes > 0.0] = cfg.reset_voltage
        self.voltage = np.clip(self.voltage, *cfg.voltage_clip).astype(np.float32)
        self.trace = (cfg.trace_decay * self.trace + self.spikes).astype(np.float32)
        return self.spikes

    def filtered_rate(self):
        return (1.0 - self.config.trace_decay) * self.trace

    def features(self, input_vector):
        cfg = self.config
        return np.concatenate(
            [
                self.filtered_rate(),
                self.voltage / max(1e-6, cfg.voltage_feature_scale),
                np.asarray(input_vector, dtype=np.float32),
                np.ones(1, dtype=np.float32),
            ]
        ).astype(np.float32)

    def frame(self):
        rate = self.filtered_rate()
        side = int(np.sqrt(self.config.n_reservoir))
        if side * side == self.config.n_reservoir:
            return rate.reshape(side, side)
        return rate[None, :]

    def stats(self):
        rate = self.filtered_rate()
        return {
            "instant_spike_fraction": float(np.mean(self.spikes)),
            "filtered_rate_mean": float(np.mean(rate)),
            "filtered_rate_std": float(np.std(rate)),
            "voltage_mean": float(np.mean(self.voltage)),
            "voltage_std": float(np.std(self.voltage)),
            "inhibitory_fraction": float(np.mean(self.neuron_signs < 0.0)),
        }


## Linear Actor-Critic Readout

The actor is a softmax linear policy over reservoir features. The critic is a linear value estimate over the same features. The update is an episode-level policy-gradient step with a learned value baseline.


In [ ]:
class ReservoirActorCritic:
    def __init__(self, n_features: int, config: RLConfig):
        self.n_features = int(n_features)
        self.config = replace(config)
        self.rng = np.random.default_rng(config.seed + 2)
        self.actor_W = self.rng.normal(0.0, 1e-3, size=(self.n_features, 2)).astype(np.float32)
        self.value_W = np.zeros(self.n_features, dtype=np.float32)
        self.updates = 0
        self.last_value_loss = None
        self.last_policy_entropy = None
        self.last_episode_advantage_mean = None

    @property
    def n_trained_parameters(self):
        return int(self.actor_W.size + self.value_W.size)

    def reset_weights(self, seed: int | None = None):
        if seed is not None:
            self.rng = np.random.default_rng(int(seed))
        self.actor_W = self.rng.normal(0.0, 1e-3, size=self.actor_W.shape).astype(np.float32)
        self.value_W.fill(0.0)
        self.updates = 0
        self.last_value_loss = None
        self.last_policy_entropy = None
        self.last_episode_advantage_mean = None

    def logits(self, features):
        F = np.asarray(features, dtype=np.float32)
        if F.ndim == 1:
            F = F[None, :]
        return F @ self.actor_W

    def probabilities(self, features):
        return softmax(self.logits(features))

    def value(self, features):
        F = np.asarray(features, dtype=np.float32)
        return F @ self.value_W

    def choose_action(self, features, stochastic: bool = True):
        probs = self.probabilities(features)[0]
        if stochastic:
            action = int(self.rng.choice(2, p=probs))
        else:
            action = int(np.argmax(probs))
        return action, probs

    def update_episode(self, features, actions, rewards):
        F = np.asarray(features, dtype=np.float32)
        actions = np.asarray(actions, dtype=np.int64)
        rewards = np.asarray(rewards, dtype=np.float32)
        if len(rewards) == 0:
            return None

        returns = np.zeros_like(rewards, dtype=np.float32)
        running_return = 0.0
        for idx in range(len(rewards) - 1, -1, -1):
            running_return = rewards[idx] + self.config.gamma * running_return
            returns[idx] = running_return

        values = F @ self.value_W
        advantages = returns - values
        actor_advantages = advantages
        if self.config.normalize_advantage and len(actor_advantages) > 1:
            actor_advantages = (actor_advantages - actor_advantages.mean()) / (actor_advantages.std() + 1e-6)

        probs = self.probabilities(F)
        selected = np.zeros_like(probs)
        selected[np.arange(len(actions)), actions] = 1.0
        grad_logits = (probs - selected) * actor_advantages[:, None]

        entropy = -np.sum(probs * np.log(probs + 1e-8), axis=1)
        entropy_grad = probs * (np.log(probs + 1e-8) + entropy[:, None])
        grad_logits += self.config.entropy_beta * entropy_grad

        grad_actor = F.T @ grad_logits / len(actions)
        grad_actor += self.config.l2 * self.actor_W
        grad_value = F.T @ (values - returns) / len(actions)
        grad_value += self.config.l2 * self.value_W

        self.actor_W -= self.config.actor_lr * grad_actor.astype(np.float32)
        self.value_W -= self.config.critic_lr * grad_value.astype(np.float32)
        self.actor_W = np.clip(self.actor_W, -10.0, 10.0)
        self.value_W = np.clip(self.value_W, -100.0, 100.0)

        self.updates += 1
        self.last_value_loss = float(np.mean((values - returns) ** 2))
        self.last_policy_entropy = float(np.mean(entropy))
        self.last_episode_advantage_mean = float(np.mean(advantages))
        return {
            "value_loss": self.last_value_loss,
            "entropy": self.last_policy_entropy,
            "advantage_mean": self.last_episode_advantage_mean,
        }


## Episode And Training Loops

`run_episode` is shared by batch training, evaluation, and the live rollout view. Training stores the reservoir feature at each decision and applies one actor-critic update at the end of the episode.


In [ ]:
def make_system(reservoir_config: ReservoirConfig = RESERVOIR_CONFIG, rl_config: RLConfig = RL_CONFIG):
    env = make_env(rl_config, seed=rl_config.seed)
    reservoir = Reservoir(reservoir_config)
    agent = ReservoirActorCritic(reservoir.n_features, rl_config)
    return env, reservoir, agent


def run_episode(
    env,
    reservoir,
    agent: ReservoirActorCritic,
    config: RLConfig,
    train: bool = True,
    stochastic: bool = True,
    seed: int | None = None,
    render_callback=None,
    delay_s: float = 0.0,
):
    observation, _ = env.reset(seed=seed)
    reservoir.reset()
    previous_action = 0.0
    features, actions, rewards = [], [], []
    total_reward = 0.0
    last_probs = np.full(2, 0.5, dtype=np.float32)
    terminated_flag = False
    truncated_flag = False

    for step in range(config.max_steps):
        input_vector = encode_cartpole_observation(
            env,
            observation,
            previous_action=previous_action,
            step_fraction=step / max(1, config.max_steps),
        )
        reservoir.step(input_vector)
        feature = reservoir.features(input_vector)
        action, probs = agent.choose_action(feature, stochastic=stochastic)
        last_probs = probs

        next_observation, reward, terminated, truncated, _ = env.step(action)
        terminated_flag = bool(terminated)
        truncated_flag = bool(truncated)
        done = bool(terminated_flag or truncated_flag)
        scaled_reward = float(reward) * config.reward_scale

        features.append(feature)
        actions.append(action)
        rewards.append(scaled_reward)
        total_reward += float(reward)

        if render_callback is not None:
            render_callback(
                observation=next_observation,
                action=action,
                probs=probs,
                step=step + 1,
                total_reward=total_reward,
                done=done,
            )
            if delay_s > 0.0:
                time.sleep(delay_s)

        previous_action = 1.0 if action == 1 else -1.0
        observation = next_observation
        if done:
            break

    metrics = None
    if train:
        metrics = agent.update_episode(features, actions, rewards)
    return {
        "return": float(total_reward),
        "steps": len(rewards),
        "terminated": terminated_flag,
        "truncated": truncated_flag,
        "last_probs": last_probs,
        "metrics": metrics,
    }


def evaluate_agent(env, reservoir, agent, config: RLConfig, episodes: int | None = None, seed_offset: int = 10_000):
    episodes = config.eval_episodes if episodes is None else int(episodes)
    returns = []
    for idx in range(episodes):
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=False,
            stochastic=False,
            seed=seed_offset + idx,
        )
        returns.append(result["return"])
    return np.asarray(returns, dtype=np.float32)


def train_agent(
    env,
    reservoir,
    agent,
    config: RLConfig,
    episodes: int | None = None,
    eval_every: int | None = None,
    progress: bool = True,
):
    episodes = config.train_episodes if episodes is None else int(episodes)
    eval_every = config.eval_every if eval_every is None else int(eval_every)
    returns = []
    eval_history = []
    iterator = range(episodes)
    if progress:
        iterator = tqdm(iterator, total=episodes, desc="actor-critic episodes")
    for episode in iterator:
        result = run_episode(
            env,
            reservoir,
            agent,
            config,
            train=True,
            stochastic=True,
            seed=config.seed * 100_000 + episode,
        )
        returns.append(result["return"])
        if eval_every and (episode + 1) % eval_every == 0:
            eval_returns = evaluate_agent(
                env,
                reservoir,
                agent,
                config,
                episodes=max(10, min(30, config.eval_episodes)),
                seed_offset=20_000 + episode * 100,
            )
            eval_history.append((episode + 1, float(eval_returns.mean()), float(eval_returns.min()), float(eval_returns.max())))
            if progress:
                iterator.set_postfix(
                    train_last50=f"{np.mean(returns[-50:]):.1f}",
                    eval=f"{eval_returns.mean():.1f}",
                )
    return {
        "returns": np.asarray(returns, dtype=np.float32),
        "eval_history": eval_history,
    }


## Build And Train

The LIF defaults use a filtered-spike plus membrane-voltage readout. In local tests the default run learned clearly non-random greedy policies and was somewhat stronger than the MPR version, though still below a tuned deep RL baseline.


In [ ]:
env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)
print(f"reservoir neurons: {RESERVOIR_CONFIG.n_reservoir}")
print(f"feature dimension: {reservoir.n_features}")
print(f"trained parameters: {agent.n_trained_parameters}")
print("reservoir stats:", reservoir.stats())

start_time = time.perf_counter()
history = train_agent(env, reservoir, agent, RL_CONFIG)
train_seconds = time.perf_counter() - start_time

eval_returns = evaluate_agent(env, reservoir, agent, RL_CONFIG)
returns = history["returns"]
print(f"training time: {train_seconds:.2f}s")
print(f"train last 50: {returns[-50:].mean():.1f}")
print(f"train best 100: {max(np.mean(returns[max(0, idx - 100):idx]) for idx in range(100, len(returns) + 1)):.1f}")
print(f"greedy eval mean/min/max over {len(eval_returns)} episodes: {eval_returns.mean():.1f} / {eval_returns.min():.1f} / {eval_returns.max():.1f}")
print(f"last value loss: {agent.last_value_loss:.6f}; policy entropy: {agent.last_policy_entropy:.4f}")
print("final reservoir stats:", reservoir.stats())


## Training Curves

The policy is stochastic during training, so the raw returns are noisy. The greedy evaluation checkpoints are a better indicator of whether the readout has learned a useful controller.


In [ ]:
def plot_training_history(history, eval_returns=None):
    returns = np.asarray(history["returns"], dtype=np.float32)
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=returns, mode="lines", name="train return", line=dict(width=1, color="#9ecae1")))
    fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="train MA50", line=dict(width=3, color="#1f77b4")))
    if history["eval_history"]:
        episodes, means, mins, maxs = zip(*history["eval_history"])
        fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="greedy eval mean", line=dict(width=3, color="#f58518")))
        fig.add_trace(go.Scatter(x=episodes, y=maxs, mode="markers", name="greedy eval max", marker=dict(color="#54a24b", size=7)))
    if eval_returns is not None:
        fig.add_hline(y=float(np.mean(eval_returns)), line_dash="dot", line_color="#e45756", annotation_text="final eval mean")
    fig.update_layout(
        title="CartPole reservoir actor-critic training",
        xaxis_title="episode",
        yaxis_title="return before failure or 500-step cap",
        width=950,
        height=430,
        margin=dict(l=50, r=20, t=60, b=45),
    )
    fig.show()
    return fig

training_fig = plot_training_history(history, eval_returns)


## CartPole Drawing Helpers

The live view draws CartPole directly from the environment state, so it works without pygame or Gym render dependencies.


In [ ]:
def cartpole_traces(observation, env):
    x, _, theta, _ = [float(v) for v in observation]
    pole_length = float(env.unwrapped.length) * 2.0
    cart_half_width = 0.18
    cart_half_height = 0.08
    pivot_y = cart_half_height
    tip_x = x + pole_length * np.sin(theta)
    tip_y = pivot_y + pole_length * np.cos(theta)
    cart_x = [x - cart_half_width, x + cart_half_width, x + cart_half_width, x - cart_half_width, x - cart_half_width]
    cart_y = [-cart_half_height, -cart_half_height, cart_half_height, cart_half_height, -cart_half_height]
    return cart_x, cart_y, [x, tip_x], [pivot_y, tip_y]


def make_cartpole_figure(env):
    observation, _ = env.reset(seed=12345)
    cart_x, cart_y, pole_x, pole_y = cartpole_traces(observation, env)
    xlim = float(env.unwrapped.x_threshold) + 0.4
    fig = go.FigureWidget()
    fig.add_trace(go.Scatter(x=[-xlim, xlim], y=[-0.09, -0.09], mode="lines", name="track", line=dict(color="#555", width=2), showlegend=False))
    fig.add_trace(go.Scatter(x=cart_x, y=cart_y, mode="lines", fill="toself", name="cart", line=dict(color="#1f77b4", width=2), fillcolor="rgba(31,119,180,0.35)", showlegend=False))
    fig.add_trace(go.Scatter(x=pole_x, y=pole_y, mode="lines+markers", name="pole", line=dict(color="#f58518", width=5), marker=dict(size=[8, 10]), showlegend=False))
    fig.update_layout(
        width=620,
        height=330,
        margin=dict(l=20, r=20, t=32, b=25),
        title=dict(text="CartPole rollout", x=0.5, font=dict(size=14)),
        xaxis=dict(range=[-xlim, xlim], zeroline=False, fixedrange=True),
        yaxis=dict(range=[-0.25, 1.15], scaleanchor="x", scaleratio=1, zeroline=False, fixedrange=True),
    )
    return fig


def update_cartpole_figure(fig, observation, env):
    cart_x, cart_y, pole_x, pole_y = cartpole_traces(observation, env)
    with fig.batch_update():
        fig.data[1].x = cart_x
        fig.data[1].y = cart_y
        fig.data[2].x = pole_x
        fig.data[2].y = pole_y


## Live RL Workbench

Use this after the training cell. You can keep training the same actor-critic readout, evaluate it, or watch a greedy rollout. Reservoir weights are fixed; reset agent only clears the trainable actor and critic readouts.


In [ ]:
class CartPoleRLWorkbench:
    def __init__(self, env, reservoir, agent: ReservoirActorCritic, config: RLConfig, history=None):
        self.env = env
        self.reservoir = reservoir
        self.agent = agent
        self.config = replace(config)
        self.history = {"returns": [], "eval_history": []} if history is None else {
            "returns": list(np.asarray(history["returns"], dtype=float)),
            "eval_history": list(history.get("eval_history", [])),
        }
        self._stop = threading.Event()
        self._thread = None
        self._lock = threading.RLock()

        self.world_fig = make_cartpole_figure(env)
        self.policy_fig = go.FigureWidget(data=[go.Bar(x=["left", "right"], y=[0.5, 0.5], marker_color=["#4c78a8", "#4c78a8"])])
        self.policy_fig.update_layout(width=310, height=260, yaxis=dict(range=[0, 1], fixedrange=True), margin=dict(l=35, r=10, t=32, b=35), title=dict(text="policy probabilities", x=0.5, font=dict(size=14)))
        self.reservoir_fig = go.FigureWidget(data=[go.Heatmap(z=self.reservoir.frame(), colorscale="RdBu", zmin=-1, zmax=1, showscale=False)])
        self.reservoir_fig.update_layout(width=310, height=310, margin=dict(l=0, r=0, t=32, b=0), title=dict(text=self.reservoir.frame_title, x=0.5, font=dict(size=14)), xaxis=dict(visible=False), yaxis=dict(visible=False, autorange="reversed"))
        self.return_fig = go.FigureWidget()
        self.return_fig.update_layout(width=950, height=320, margin=dict(l=45, r=20, t=35, b=40), xaxis_title="episode", yaxis_title="return", title=dict(text="live training returns", x=0.5, font=dict(size=14)))
        self._refresh_return_figure()

        self.train_button = widgets.Button(description="Train", icon="graduation-cap", button_style="success")
        self.eval_button = widgets.Button(description="Eval", icon="check", button_style="info")
        self.rollout_button = widgets.Button(description="Rollout", icon="play", button_style="primary")
        self.stop_button = widgets.Button(description="Stop", icon="stop", button_style="danger")
        self.reset_agent_button = widgets.Button(description="Reset Agent", icon="eraser")

        self.train_episodes = widgets.IntSlider(value=100, min=1, max=3000, step=1, description="episodes", continuous_update=False)
        self.eval_episodes = widgets.IntSlider(value=20, min=1, max=100, step=1, description="eval n", continuous_update=False)
        self.delay_ms = widgets.IntSlider(value=20, min=0, max=200, step=5, description="delay ms", continuous_update=False)
        self.stochastic_rollout = widgets.Checkbox(value=False, description="stochastic rollout")
        self.actor_lr = widgets.FloatLogSlider(value=self.agent.config.actor_lr, base=10, min=-6, max=-1, step=0.1, description="actor lr", continuous_update=False, style={"description_width": "initial"})
        self.critic_lr = widgets.FloatLogSlider(value=self.agent.config.critic_lr, base=10, min=-5, max=0, step=0.1, description="critic lr", continuous_update=False, style={"description_width": "initial"})
        self.gamma = widgets.FloatSlider(value=self.agent.config.gamma, min=0.90, max=0.999, step=0.001, readout_format=".3f", description="gamma", continuous_update=False)
        self.reward_scale = widgets.FloatLogSlider(value=self.agent.config.reward_scale, base=10, min=-4, max=0, step=0.1, description="reward scale", continuous_update=False, style={"description_width": "initial"})
        self.status = widgets.HTML(value="idle")

        self.train_button.on_click(lambda _: self.train_async())
        self.eval_button.on_click(lambda _: self.evaluate_async())
        self.rollout_button.on_click(lambda _: self.rollout_async())
        self.stop_button.on_click(lambda _: self.stop())
        self.reset_agent_button.on_click(lambda _: self.reset_agent())

    def _sync_config(self):
        self.agent.config.actor_lr = float(self.actor_lr.value)
        self.agent.config.critic_lr = float(self.critic_lr.value)
        self.agent.config.gamma = float(self.gamma.value)
        self.agent.config.reward_scale = float(self.reward_scale.value)
        self.config = replace(
            self.config,
            actor_lr=float(self.actor_lr.value),
            critic_lr=float(self.critic_lr.value),
            gamma=float(self.gamma.value),
            reward_scale=float(self.reward_scale.value),
        )

    def _refresh_return_figure(self):
        returns = np.asarray(self.history["returns"], dtype=np.float32)
        with self.return_fig.batch_update():
            self.return_fig.data = []
            if len(returns):
                self.return_fig.add_trace(go.Scatter(y=returns, mode="lines", name="return", line=dict(color="#9ecae1", width=1)))
                self.return_fig.add_trace(go.Scatter(y=moving_average(returns, 50), mode="lines", name="MA50", line=dict(color="#1f77b4", width=3)))
            if self.history["eval_history"]:
                episodes, means, mins, maxs = zip(*self.history["eval_history"])
                self.return_fig.add_trace(go.Scatter(x=episodes, y=means, mode="lines+markers", name="eval mean", line=dict(color="#f58518", width=3)))

    def _draw_policy(self, probs):
        pred = int(np.argmax(probs))
        colors = ["#4c78a8", "#4c78a8"]
        colors[pred] = "#f58518"
        with self.policy_fig.batch_update():
            self.policy_fig.data[0].y = probs
            self.policy_fig.data[0].marker.color = colors
        with self.reservoir_fig.batch_update():
            self.reservoir_fig.data[0].z = self.reservoir.frame()

    def _render_callback(self, observation, action, probs, step, total_reward, done):
        with self._lock:
            update_cartpole_figure(self.world_fig, observation, self.env)
            self._draw_policy(probs)
            self.status.value = f"rollout step={step} return={total_reward:.0f} action={action} done={done}"

    def train(self, episodes: int | None = None):
        self._sync_config()
        episodes = int(self.train_episodes.value if episodes is None else episodes)
        start_count = len(self.history["returns"])
        for idx in range(episodes):
            if self._stop.is_set():
                break
            result = run_episode(
                self.env,
                self.reservoir,
                self.agent,
                self.agent.config,
                train=True,
                stochastic=True,
                seed=self.config.seed * 100_000 + start_count + idx,
            )
            self.history["returns"].append(result["return"])
            if (idx + 1) % 10 == 0 or idx == episodes - 1:
                returns = np.asarray(self.history["returns"], dtype=np.float32)
                ma50 = float(np.mean(returns[-50:])) if len(returns) else np.nan
                with self._lock:
                    self._refresh_return_figure()
                    self.status.value = f"trained={len(returns)} last={result['return']:.0f} MA50={ma50:.1f} updates={self.agent.updates}"
        self._stop.clear()

    def evaluate(self):
        self._sync_config()
        with self._lock:
            self.status.value = "evaluating..."
        returns = evaluate_agent(
            self.env,
            self.reservoir,
            self.agent,
            self.agent.config,
            episodes=int(self.eval_episodes.value),
            seed_offset=50_000 + len(self.history["returns"]) * 100,
        )
        self.history["eval_history"].append((len(self.history["returns"]), float(returns.mean()), float(returns.min()), float(returns.max())))
        with self._lock:
            self._refresh_return_figure()
            self.status.value = f"eval mean/min/max={returns.mean():.1f}/{returns.min():.0f}/{returns.max():.0f}"

    def rollout(self):
        self._sync_config()
        self._stop.clear()
        result = run_episode(
            self.env,
            self.reservoir,
            self.agent,
            self.agent.config,
            train=False,
            stochastic=bool(self.stochastic_rollout.value),
            seed=80_000 + len(self.history["returns"]),
            render_callback=self._render_callback,
            delay_s=float(self.delay_ms.value) / 1000.0,
        )
        with self._lock:
            self.status.value = f"rollout return={result['return']:.0f} steps={result['steps']}"

    def _start_thread(self, target):
        self.stop()
        self._stop.clear()
        self._thread = threading.Thread(target=target, daemon=True)
        self._thread.start()

    def train_async(self):
        self._start_thread(self.train)

    def evaluate_async(self):
        self._start_thread(self.evaluate)

    def rollout_async(self):
        self._start_thread(self.rollout)

    def stop(self):
        self._stop.set()
        if self._thread is not None and self._thread.is_alive():
            self._thread.join(timeout=2)

    def reset_agent(self):
        self.stop()
        with self._lock:
            self.agent.reset_weights(seed=self.config.seed + self.agent.updates + 10)
            self.history = {"returns": [], "eval_history": []}
            self._refresh_return_figure()
            self.status.value = "agent reset"

    def display(self):
        controls = widgets.VBox([
            widgets.HBox([self.train_button, self.eval_button, self.rollout_button, self.stop_button, self.reset_agent_button]),
            widgets.HBox([self.train_episodes, self.eval_episodes, self.delay_ms, self.stochastic_rollout]),
            widgets.HBox([self.actor_lr, self.critic_lr, self.gamma, self.reward_scale]),
            self.status,
        ])
        display(widgets.VBox([
            controls,
            widgets.HBox([self.world_fig, widgets.VBox([self.policy_fig, self.reservoir_fig])]),
            self.return_fig,
        ]))


try:
    cartpole_workbench.stop()
except NameError:
    pass

cartpole_workbench = CartPoleRLWorkbench(env, reservoir, agent, RL_CONFIG, history)
cartpole_workbench.display()


## Manual Experiments

Small edits that are useful while testing.


In [ ]:
# Stop background work:
# cartpole_workbench.stop()

# Train longer from the current weights:
# more_history = train_agent(env, reservoir, agent, RL_CONFIG, episodes=1000, eval_every=200)
# history["returns"] = np.concatenate([history["returns"], more_history["returns"]])
# history["eval_history"].extend(more_history["eval_history"])
# plot_training_history(history, evaluate_agent(env, reservoir, agent, RL_CONFIG))

# Rebuild after changing reservoir parameters:
# RESERVOIR_CONFIG = replace(RESERVOIR_CONFIG, n_reservoir=512, seed=5)
# env, reservoir, agent = make_system(RESERVOIR_CONFIG, RL_CONFIG)

# Change RL hyperparameters before rerunning training:
# RL_CONFIG = replace(RL_CONFIG, train_episodes=3000, actor_lr=1e-4, critic_lr=1e-2)


# LIF-specific toggles:
# RESERVOIR_CONFIG = replace(RESERVOIR_CONFIG, threshold=0.9, input_scale=1.2)
# To test a stricter spike-only readout, edit Reservoir.n_features and Reservoir.features to drop voltage.
